# LeetCode #90: Subsets II

https://leetcode.com/problems/subsets-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^n \cdot n)$ | $O(2^n \cdot n)$ |
| **Optimal: Backtracking + Deduplication ★** | $O(2^n \cdot n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Generate all $2^n$ subsets, insert each into a hash set of sorted tuples to remove duplicates, then convert to a list. Works but builds and discards many duplicate subsets before de-duplication.

### Optimal: Backtracking + Deduplication ★
Sort the array so duplicates are adjacent. At each recursion level, skip element `i` if it equals the previous element AND `i > start` (meaning we already explored a branch starting with that value at this depth). This prunes entire duplicate subtrees without any set.

**Why this is better than Brute Force:** Sorting lets the skip condition fire in $O(1)$ per candidate, cutting duplicate branches before they generate any subsets, so memory stays $O(n)$ instead of $O(2^n \cdot n)$.

**Constraints:**
* 1 <= nums.length <= 10
* -10 <= nums[i] <= 10

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<IList<int>> SubsetsWithDup(int[] nums) {
        Array.Sort(nums);
        var result = new List<IList<int>>();
        Backtrack(nums, 0, new List<int>(), result);
        return result;
    }

    private void Backtrack(int[] nums, int start, List<int> path, IList<IList<int>> result) {
        // Every state of the path is a valid subset — record it
        result.Add(new List<int>(path));
        for (int i = start; i < nums.Length; i++) {
            // Skip duplicate elements at the same recursion level to avoid duplicate subsets
            if (i > start && nums[i] == nums[i - 1]) continue;
            path.Add(nums[i]);
            Backtrack(nums, i + 1, path, result);
            // Undo the choice to explore subsets that omit nums[i]
            path.RemoveAt(path.Count - 1);
        }
    }
}

### Python

In [ ]:
class Solution:
    def subsetsWithDup(self, nums: list[int]) -> list[list[int]]:
        nums.sort()
        result = []

        def backtrack(start: int, path: list[int]) -> None:
            # Every state of the path is a valid subset — record it
            result.append(list(path))
            for i in range(start, len(nums)):
                # Skip duplicate elements at the same recursion level to avoid duplicate subsets
                if i > start and nums[i] == nums[i - 1]:
                    continue
                path.append(nums[i])
                backtrack(i + 1, path)
                # Undo the choice to explore subsets that omit nums[i]
                path.pop()

        backtrack(0, [])
        return result

### Go

In [ ]:
func subsetsWithDup(nums []int) [][]int {
    sort.Ints(nums)
    result := [][]int{}
    path := []int{}

    var backtrack func(start int)
    backtrack = func(start int) {
        // Every state of the path is a valid subset — record it
        subset := make([]int, len(path))
        copy(subset, path)
        result = append(result, subset)
        for i := start; i < len(nums); i++ {
            // Skip duplicate elements at the same recursion level to avoid duplicate subsets
            if i > start && nums[i] == nums[i-1] {
                continue
            }
            path = append(path, nums[i])
            backtrack(i + 1)
            // Undo the choice to explore subsets that omit nums[i]
            path = path[:len(path)-1]
        }
    }
    backtrack(0)
    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn subsets_with_dup(mut nums: Vec<i32>) -> Vec<Vec<i32>> {
        nums.sort();
        let mut result = Vec::new();
        let mut path = Vec::new();
        Self::backtrack(&nums, 0, &mut path, &mut result);
        result
    }

    fn backtrack(nums: &[i32], start: usize, path: &mut Vec<i32>, result: &mut Vec<Vec<i32>>) {
        // Every state of the path is a valid subset — record it
        result.push(path.clone());
        for i in start..nums.len() {
            // Skip duplicate elements at the same recursion level to avoid duplicate subsets
            if i > start && nums[i] == nums[i - 1] { continue; }
            path.push(nums[i]);
            Self::backtrack(nums, i + 1, path, result);
            // Undo the choice to explore subsets that omit nums[i]
            path.pop();
        }
    }
}

## Example Scenarios

**1. Common Case**
**Input:** nums = [1, 2, 2]
After sorting: [1, 2, 2]. Records []. Adds 1 → records [1]; from i=1, adds 2 → [1,2]; adds 2 again (i=2, i>start=1, nums[2]==nums[1]) → **skipped**. Records [1,2]. Backtracks, adds 2 (i=1) → [2]; from i=2, skip (duplicate). Adds 2 (i=2, i==start=2) → [2,2]. Result: [[], [1], [1,2], [2], [2,2]] — 5 unique subsets (not 8 from brute force).

**2. Slightly Complex**
**Input:** nums = [1, 1, 2, 2]
Sorted: [1, 1, 2, 2]. Unique subsets: [], [1], [1,1], [1,1,2], [1,1,2,2], [1,2], [1,2,2], [2], [2,2] — 9 subsets. The brute-force $2^4=16$ would include 7 duplicates that the skip guard prunes.

**3. Edge Case: Time Factor**
**Input:** nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] (all distinct, n=10)
No duplicates means the skip guard never fires; behavior is identical to #78 (Subsets), generating all $2^{10}=1024$ subsets with no wasted calls.

**4. Edge Case: Space Factor**
**Input:** nums = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] (ten identical elements, n=10)
Only 11 unique subsets exist (size 0 through 10). The skip guard fires at every level after the first choice, collapsing the $2^{10}=1024$ brute-force paths into exactly 11 explored branches. Peak path length is 10; stack depth is 10. Negligible memory.

**5. Almost-Impossible but Plausible**
**Input:** nums = [-10, -10, 0, 0, 10, 10]
Six elements with three pairs of duplicates. $2^6/8 = 8$... actually $\prod (k_i+1) = 3 \times 3 \times 3 = 27$ unique subsets. The skip guard fires for each duplicate pair at each recursion level, reducing explored branches from 64 to 27. Negative values are handled identically to positives.